In [1]:
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal

In [2]:
reference_file = 'ref_data/dataset-arb_data-allPhysio_detail-2clusterStandardizedChangeScores.csv'
wave1_file = 'data/dataset-ariWave1_data-allPhysio_detail-standardizedChangeScores.csv'
wave2_file = 'data/dataset-ariWave2_data-allPhysio_detail-standardizedChangeScores.csv'

ref_df = pd.read_csv(reference_file)
wave1_df = pd.read_csv(wave1_file)
wave2_df = pd.read_csv(wave2_file)

In [3]:
all_relevant_metrics = ['SCR', 'SCL', 'IBI', 'RSA',
                         'MAP', 'CO', 'PEP', 'TPR']
relevant_columns = [m + '_delta' for m in all_relevant_metrics]
wave1_df = wave1_df[['subject', 'trial'] +  relevant_columns]
relevant_ref_df = ref_df[ref_df['metric'].isin(all_relevant_metrics)]
relevant_ref_df = relevant_ref_df.reset_index(drop=True)

n_subjects = len(np.unique(wave1_df['subject'].values))

In [4]:
wave1_df

,subject,trial,SCR_delta,SCL_delta,IBI_delta,RSA_delta,MAP_delta,CO_delta,PEP_delta,TPR_delta
0,sub-003,1,-1.108376,-0.680116,-0.413194,0.313878,0.472260,0.925435,-1.367417,-0.188928
1,sub-003,2,-0.072717,-0.725578,1.064173,-0.029487,0.068875,-0.243170,-0.806433,0.035395
2,sub-003,3,0.617723,-0.752049,0.166406,-0.345952,-0.108803,-0.165196,-1.367417,-0.119795
3,sub-003,4,0.272503,-0.658250,2.033700,0.449909,0.027823,-0.764007,-0.740435,0.258160
4,sub-006,1,0.893899,0.766347,-2.586264,-4.264711,-0.041156,0.847683,-2.812776,-0.621452
...,...,...,...,...,...,...,...,...,...,...
179,sub-067,4,1.377207,0.680878,-0.121121,0.098942,0.125406,0.881853,-0.311447,-0.110413
180,sub-068,1,0.272503,0.414029,-1.695569,1.313430,-0.368587,0.839570,-0.945029,-0.022833
181,sub-068,2,-0.417936,0.032347,-1.141431,0.421705,-0.378918,1.587217,-0.384045,-0.079501
182,sub-068,3,-0.763156,0.056894,-0.560364,1.832569,-1.360200,1.038062,-0.516041,-0.270592


In [5]:
reference_trials = [1, 2, 3, 4]
trials = [1, 2, 3, 4]

challenge_scores_ntr = {key: np.zeros((len(trials), len(reference_trials))) for key in wave1_df['subject']}
threat_scores_ntr = {key: np.zeros((len(trials), len(reference_trials))) for key in wave1_df['subject']}


In [9]:
# calculate log-likelihoods
for r in reference_trials:
    for t in trials:
        wave1_temp_df = wave1_df[wave1_df['trial'] == t]
        ref_temp_df = relevant_ref_df[relevant_ref_df['trial'] == r]
        for sub in wave1_df['subject']:
            sub_df = wave1_temp_df[wave1_temp_df['subject'] == sub]            
            challenge_means = ref_temp_df['challenge_cluster_means'].values
            threat_means = ref_temp_df['threat_cluster_means'].values
            challenge_std = np.ones_like(challenge_means)
            threat_std = np.ones_like(threat_means)
            
            ## calculate the dual likelihoods
            challenge_pdf = multivariate_normal(challenge_means, np.diag(challenge_std))
            challenge_score = np.nan_to_num(challenge_pdf.pdf(sub_df[relevant_columns]), nan=0)
            challenge_scores_ntr[sub][r - 1, t - 1] = challenge_score
            
            threat_pdf = multivariate_normal(threat_means, np.diag(threat_std))
            threat_score = np.nan_to_num(threat_pdf.pdf(sub_df[relevant_columns]), nan=0)
            threat_scores_ntr[sub][r - 1, t - 1] = threat_score

In [10]:
# ensemble trials to create a single trajectory per participant across ref trials
challenge_scores_nr_means_std = {key: np.zeros((len(trials), 2)) for key in wave1_df['subject']}
threat_scores_nr_means_std = {key: np.zeros((len(trials), 2)) for key in wave1_df['subject']}

for sub in wave1_df['subject']:
    challenge_scores_nr_means_std[sub][:, 0] = challenge_scores_ntr[sub].mean(axis = -1)
    challenge_scores_nr_means_std[sub][:, 1] = challenge_scores_ntr[sub].std(axis = -1)
    
    threat_scores_nr_means_std[sub][:, 0] = threat_scores_ntr[sub].mean(axis = -1)
    threat_scores_nr_means_std[sub][:, 1] = threat_scores_ntr[sub].std(axis = -1)

In [11]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

time_labels = ["T1", "T2", "T3", "T4"]
marker_symbols = ["circle", "square", "diamond", "x"]

fig = go.Figure()

for subject in wave1_df['subject']:
    arr_x = challenge_scores_nr_means_std[subject]
    arr_y = threat_scores_nr_means_std[subject]

    if arr_x.shape != arr_y.shape:
        raise ValueError(f"Shape mismatch for {subject}: {arr_x.shape} vs {arr_y.shape}")
    if arr_x.shape[1] != 2:
        raise ValueError(f"Expected shape (n_timepoints, 2) for {subject}, got {arr_x.shape}")

    x_means = arr_x[:, 0]
    x_stds  = arr_x[:, 1]
    y_means = arr_y[:, 0]
    y_stds  = arr_y[:, 1]

    # line connecting the trajectory
    fig.add_trace(
        go.Scatter(
            x=x_means,
            y=y_means,
            mode="lines",
            name=subject,
            legendgroup=subject,
            showlegend=True,
            hoverinfo="skip"
        )
    )

    # one marker per timepoint so marker symbol can vary
    for i, (xm, xs, ym, ys) in enumerate(zip(x_means, x_stds, y_means, y_stds)):
        fig.add_trace(
            go.Scatter(
                x=[xm],
                y=[ym],
                mode="markers",
                name=subject if i == 0 else None,
                legendgroup=subject,
                showlegend=False,
                marker=dict(
                    size=10,
                    symbol=marker_symbols[i % len(marker_symbols)]
                ),
                error_x=dict(
                    type="data",
                    array=[xs],
                    visible=True
                ),
                error_y=dict(
                    type="data",
                    array=[ys],
                    visible=True
                ),
                customdata=[[subject, time_labels[i], xs, ys]],
                hovertemplate=(
                    "Subject: %{customdata[0]}<br>"
                    "Time: %{customdata[1]}<br>"
                    "X mean: %{x:.3f}<br>"
                    "X std: %{customdata[2]:.3f}<br>"
                    "Y mean: %{y:.3f}<br>"
                    "Y std: %{customdata[3]:.3f}"
                    "<extra></extra>"
                )
            )
        )

fig.update_layout(
    title="2D Trajectories with X/Y Error Bars",
    xaxis_title="X mean",
    yaxis_title="Y mean",
    template="plotly_white",
    hovermode="closest"
)

fig.show()